In [112]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression,LinearRegression


from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer,MissingIndicator

In [113]:
df = pd.read_csv('student_studydata.csv')

In [114]:
df.sample(5)

,Student_ID,Age,Gender,Study_Hours,Attendance,Math_Score,Science_Score,English_Score,Family_Income,Internet_Access
4,5,19,Female,7.0,98.0,95.0,96.0,NaN,60000.0,Yes
23,24,18,Male,3.0,78.0,65.0,67.0,NaN,32000.0,No
8,9,18,Female,5.0,90.0,85.0,NaN,84.0,48000.0,Yes
19,20,19,Male,NaN,86.0,77.0,75.0,76.0,41000.0,No
12,13,20,Female,7.0,97.0,NaN,95.0,94.0,68000.0,Yes


In [80]:
summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().sum() / len(df)) * 100
})

print(summary)

                 Missing Values  Percentage
Student_ID                    0         0.0
Age                           0         0.0
Gender                        0         0.0
Study_Hours                   4        16.0
Attendance                    3        12.0
Math_Score                    3        12.0
Science_Score                 3        12.0
English_Score                 4        16.0
Family_Income                 2         8.0
Internet_Access               2         8.0


In [81]:
imputer = SimpleImputer(strategy='mean')
df[['Study_Hours', 'Science_Score', 'English_Score']] = imputer.fit_transform(df[['Study_Hours', 'Science_Score', 'English_Score']])

In [82]:
print(df.isnull().sum())

Student_ID         0
Age                0
Gender             0
Study_Hours        0
Attendance         3
Math_Score         3
Science_Score      0
English_Score      0
Family_Income      2
Internet_Access    2
dtype: int64


In [83]:
# arbitrary_value fill missing values
imputer = SimpleImputer(strategy='constant', fill_value="not defined")
df[['Family_Income', 'Internet_Access']] = imputer.fit_transform(df[['Family_Income', 'Internet_Access']])
print(df.isnull().sum())

Student_ID         0
Age                0
Gender             0
Study_Hours        0
Attendance         3
Math_Score         3
Science_Score      0
English_Score      0
Family_Income      0
Internet_Access    0
dtype: int64


In [84]:
import numpy as np

end_value = df['Study_Hours'].mean() + 3 * df['Study_Hours'].std()

df['Study_Hours'] = df['Study_Hours'].fillna(end_value)

In [85]:
print(df.isnull().sum())

Student_ID         0
Age                0
Gender             0
Study_Hours        0
Attendance         3
Math_Score         3
Science_Score      0
English_Score      0
Family_Income      0
Internet_Access    0
dtype: int64


In [86]:
import numpy as np

columns = ['Study_Hours', 'Attendance', 'Math_Score']

for col in columns:
    missing = df[col].isnull()

    df.loc[missing, col] = df[col].dropna().sample(
        n=missing.sum(),
        replace=True,      # Important if missing values > available values
        random_state=42
    ).values

In [87]:
print(df.isnull().sum())

Student_ID         0
Age                0
Gender             0
Study_Hours        0
Attendance         0
Math_Score         0
Science_Score      0
English_Score      0
Family_Income      0
Internet_Access    0
dtype: int64


In [88]:
x=df.drop(columns=['Math_Score','Gender'])
y=df['Math_Score']

In [89]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [90]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['Internet_Access'] = le.fit_transform(df['Internet_Access'])

In [91]:
x=df.drop(columns=['Math_Score','Gender'])
y=df['Math_Score']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [92]:
mi=MissingIndicator()
mi.fit(x_train)

,"missing_values missing_values: int, float, str, np.nan or None, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`should be set to `np.nan`, since `pd.NA` will be converted to `np.nan`.",nan
,"features features: {'missing-only', 'all'}, default='missing-only'Whether the imputer mask should represent all or a subset offeatures.- If `'missing-only'` (default), the imputer mask will only represent features containing missing values during fit time.- If `'all'`, the imputer mask will represent all features.",'missing-only'
,"sparse sparse: bool or 'auto', default='auto'Whether the imputer mask format should be sparse or dense.- If `'auto'` (default), the imputer mask will be of same type as input.- If `True`, the imputer mask will be a sparse matrix.- If `False`, the imputer mask will be a numpy array.",'auto'
,"error_on_new error_on_new: bool, default=TrueIf `True`, :meth:`transform` will raise an error when there arefeatures with missing values that have no missing values in:meth:`fit`. This is applicable only when `features='missing-only'`.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](8,)","['Student_ID','Age','Study_Hours',...,'English_Score','Family_Income', 'Internet_Access']"
"features_ features_: ndarray of shape (n_missing_features,) or (n_features,)The features indices which will be returned when calling:meth:`transform`. They are computed during :meth:`fit`. If`features='all'`, `features_` is equal to `range(n_features)`.","ndarray[int64](0,)",[]
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,8


In [93]:
x_train_missing=mi.transform(x_train)
x_test_missing=mi.transform(x_test)

In [94]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [95]:
model = LogisticRegression()

In [96]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200]
}
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,          # 5-fold cross validation
    scoring='accuracy'
)

In [97]:
df = df.dropna(subset=['Math_Score'])
x = df.drop(columns=['Math_Score'])
y = df['Math_Score']

In [98]:
df['Math_Score'] = df['Math_Score'].fillna(df['Math_Score'].median())

In [99]:
df = pd.read_csv('student_studydata.csv')

# handle missing values properly first
df['Math_Score'] = df['Math_Score'].fillna(df['Math_Score'].median())
df['Study_Hours'] = df['Study_Hours'].fillna(df['Study_Hours'].median())
df['Attendance'] = df['Attendance'].fillna(df['Attendance'].median())

# fill categorical missing values
df['Internet_Access'] = df['Internet_Access'].fillna(df['Internet_Access'].mode()[0])

# convert target into classification labels
df['Math_Class'] = pd.cut(
    df['Math_Score'],
    bins=[0, 50, 75, 100],
    labels=['Low', 'Medium', 'High']
)

In [100]:
x = df.drop(columns=['Math_Score', 'Math_Class'])
x = pd.get_dummies(x, drop_first=True)

y = df['Math_Class']

In [101]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200]
}

cv = StratifiedKFold(n_splits=3)  # 🔥 reduce from 5 to 3

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy'
)

In [102]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('student_studydata.csv')

# -------------------------
# 1. Fix target
# -------------------------
df['Math_Score'] = df['Math_Score'].fillna(df['Math_Score'].median())

df['Math_Class'] = pd.cut(
    df['Math_Score'],
    bins=[0, 50, 75, 100],
    labels=['Low', 'Medium', 'High']
)

df = df.dropna(subset=['Math_Class'])

# -------------------------
# 2. Fix missing values in features
# -------------------------
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

# -------------------------
# 3. Encode categorical variables
# -------------------------
X = df.drop(columns=['Math_Score', 'Math_Class'])
X = pd.get_dummies(X, drop_first=True)

y = df['Math_Class']

In [103]:
model = LogisticRegression()

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'solver': ['liblinear', 'lbfgs'],
    'max_iter': [100, 200]
}

cv = StratifiedKFold(n_splits=3)  # safer than 5

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy'
)

grid.fit(X, y)

c:\Users\admin\Desktop\langchain + ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\admin\Desktop\langchain + ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modul

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'max_iter': [100, 200], 'solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int,

In [104]:
# 1. BEST REAL-WORLD PROJECT IDEA
# 🎯 Problem Statement (Industry-Level)
# 👉 “Student Academic Performance Prediction and Risk Detection System”
# 💡 Real-life use case:

# Schools / EdTech companies like:

# Byju’s
# Unacademy
# Coursera
# School ERP systems

# want to:

# Predict student performance early and identify students who are at risk of failing so that intervention can be given.



# 🧠 3. BEST MODEL FOR THIS PROBLEM

# Since this is classification:

# 🔥 Try these models:
# ⭐ Baseline:
# Logistic Regression
# ⭐ Better:
# Random Forest Classifier
# ⭐ Industry-level:
# XGBoost Classifier
# LightGBM

# from sklearn.ensemble import RandomForestClassifier

In [105]:
# knn multivariate
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.preprocessing import OrdinalEncoder
df = pd.read_csv('student_studydata.csv')
encoder = OrdinalEncoder()

# Encode categorical columns
df[['Gender', 'Internet_Access']] = encoder.fit_transform(
    df[['Gender', 'Internet_Access']]
)


In [106]:
imputer = KNNImputer(n_neighbors=2)

In [107]:
df_imputed = pd.DataFrame(
    imputer.fit_transform(df),
    columns=df.columns
)

print(df_imputed.columns)

Index(['Student_ID', 'Age', 'Gender', 'Study_Hours', 'Attendance',
       'Math_Score', 'Science_Score', 'English_Score', 'Family_Income',
       'Internet_Access'],
      dtype='object')


In [115]:
df['Study_Hours'] = df['Study_Hours'].fillna(df['Study_Hours'].median())
df['Attendance'] = df['Attendance'].fillna(df['Attendance'].median())
df['Family_Income'] = df['Family_Income'].fillna(df['Family_Income'].median())
df['Internet_Access'] = df['Internet_Access'].fillna(df['Internet_Access'].mode()[0])


df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Internet_Access'] = df['Internet_Access'].map({'No': 0, 'Yes': 1})

In [122]:
df = df.dropna(subset=['Performance_Level'])

In [123]:
df['Performance_Level'] = (df['Math_Score'] + df['Science_Score'] + df['English_Score']) / 3
df['Performance_Level'] = pd.cut(
    df['Performance_Level'],
    bins=[0, 60, 75, 100],
    labels=['Low', 'Medium', 'High']
)

X = df.drop(columns=[
    'Math_Score',
    'Science_Score',
    'English_Score',
    'Performance_Level'
])
y = df['Performance_Level']


In [124]:
# print(X.isnull().sum())
print(y.isnull().sum())

0


In [125]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy'
)

grid.fit(X, y)

c:\Users\admin\Desktop\langchain + ml\venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [3, 5, ...], 'min_samples_split': [2, 5], 'n_estimators': [50, 100]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose ve

In [126]:
best_model = grid.best_estimator_

print(grid.best_params_)

{'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}


In [127]:
import pickle

with open("student_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

In [128]:
with open("student_model.pkl", "rb") as f:
    model = pickle.load(f)

In [129]:
new_data = pd.DataFrame([{
    "Student_ID": 101,
    "Age": 19,
    "Gender": 1,   # Female
    "Study_Hours": 6,
    "Attendance": 90,
    "Family_Income": 50000,
    "Internet_Access": 1
}])

prediction = model.predict(new_data)

print("Predicted Performance:", prediction[0])

Predicted Performance: High
